# ♟️ Chess AI with Deep Learning (AlphaZero Style)

This notebook walks you through building a chess AI step by step:

1. **Phase 1** — Chess environment setup
2. **Phase 2** — Classical Minimax engine (baseline)
3. **Phase 3** — Neural network (value + policy heads)
4. **Phase 4** — Monte Carlo Tree Search (MCTS)
5. **Phase 5** — Self-play training loop

> 💡 **Tip:** Run each cell in order. GPU is recommended for Phase 5 training.

## 📦 Install Dependencies

In [ ]:
!pip install python-chess torch torchvision numpy matplotlib tqdm

## Phase 1 — Chess Environment

We use `python-chess` to handle all board logic: move generation, legality checks, game-over detection.

In [ ]:
import chess
import chess.svg
from IPython.display import SVG, display
import numpy as np
import random

# Create a fresh board
board = chess.Board()

# Display it
display(SVG(chess.svg.board(board, size=350)))
print(f"Turn: {'White' if board.turn else 'Black'}")
print(f"Legal moves: {list(board.legal_moves)[:5]} ... ({board.legal_moves.count()} total)")

In [ ]:
def play_random_game(max_moves=200):
    """Play a game with both sides making random moves."""
    board = chess.Board()
    moves = []
    while not board.is_game_over() and len(moves) < max_moves:
        move = random.choice(list(board.legal_moves))
        board.push(move)
        moves.append(move)
    return board, moves

board, moves = play_random_game()
print(f"Game over: {board.is_game_over()}")
print(f"Result: {board.result()}")
print(f"Moves played: {len(moves)}")
display(SVG(chess.svg.board(board, size=350)))

## Phase 2 — Classical Minimax Engine (Baseline)

Before deep learning, we build a classical engine using **Minimax with Alpha-Beta pruning** and a handcrafted evaluation function. This is what AlphaZero's neural network will *replace*.

In [ ]:
# Piece values (centipawns)
PIECE_VALUES = {
    chess.PAWN:   100,
    chess.KNIGHT: 320,
    chess.BISHOP: 330,
    chess.ROOK:   500,
    chess.QUEEN:  900,
    chess.KING:   20000
}

def evaluate_board(board):
    """Simple material + mobility evaluation."""
    if board.is_checkmate():
        return -10000 if board.turn else 10000  # Current side lost
    if board.is_stalemate() or board.is_insufficient_material():
        return 0

    score = 0
    # Material count
    for piece_type, value in PIECE_VALUES.items():
        score += len(board.pieces(piece_type, chess.WHITE)) * value
        score -= len(board.pieces(piece_type, chess.BLACK)) * value

    # Mobility bonus
    score += board.legal_moves.count() * 10 * (1 if board.turn == chess.WHITE else -1)

    return score if board.turn == chess.WHITE else -score


def minimax(board, depth, alpha, beta, maximizing):
    """Minimax with alpha-beta pruning."""
    if depth == 0 or board.is_game_over():
        return evaluate_board(board)

    if maximizing:
        max_eval = float('-inf')
        for move in board.legal_moves:
            board.push(move)
            eval_ = minimax(board, depth - 1, alpha, beta, False)
            board.pop()
            max_eval = max(max_eval, eval_)
            alpha = max(alpha, eval_)
            if beta <= alpha:
                break
        return max_eval
    else:
        min_eval = float('inf')
        for move in board.legal_moves:
            board.push(move)
            eval_ = minimax(board, depth - 1, alpha, beta, True)
            board.pop()
            min_eval = min(min_eval, eval_)
            beta = min(beta, eval_)
            if beta <= alpha:
                break
        return min_eval


def best_move_minimax(board, depth=3):
    """Find best move using minimax."""
    best_move = None
    best_val = float('-inf')
    for move in board.legal_moves:
        board.push(move)
        val = minimax(board, depth - 1, float('-inf'), float('inf'), False)
        board.pop()
        if val > best_val:
            best_val = val
            best_move = move
    return best_move


# Test it
board = chess.Board()
move = best_move_minimax(board, depth=3)
print(f"Best opening move (depth 3): {board.san(move)}")
board.push(move)
display(SVG(chess.svg.board(board, size=350)))

## Phase 3 — Neural Network (AlphaZero Architecture)

The network takes a **board state tensor** as input and outputs:
- **Value head** — how good is this position? (-1 = black wins, +1 = white wins)
- **Policy head** — probability distribution over all possible moves

### Board Encoding
We encode the board as an `8×8×18` tensor:
- 12 planes for pieces (6 piece types × 2 colors)
- 1 plane for whose turn
- 4 planes for castling rights
- 1 plane for en passant

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# All possible moves encoded as indices (from-square * 64 + to-square)
# Simple encoding: 64*64 = 4096 possible (from, to) pairs
NUM_ACTIONS = 4096


def board_to_tensor(board):
    """
    Encode a chess board as a float tensor of shape (18, 8, 8).
    Channels:
      0-5:  White pieces (P, N, B, R, Q, K)
      6-11: Black pieces (P, N, B, R, Q, K)
      12:   Turn (1=white, 0=black)
      13:   White kingside castling
      14:   White queenside castling
      15:   Black kingside castling
      16:   Black queenside castling
      17:   En passant square
    """
    tensor = np.zeros((18, 8, 8), dtype=np.float32)
    piece_types = [chess.PAWN, chess.KNIGHT, chess.BISHOP,
                   chess.ROOK, chess.QUEEN, chess.KING]

    for i, piece_type in enumerate(piece_types):
        for sq in board.pieces(piece_type, chess.WHITE):
            r, c = divmod(sq, 8)
            tensor[i, r, c] = 1.0
        for sq in board.pieces(piece_type, chess.BLACK):
            r, c = divmod(sq, 8)
            tensor[i + 6, r, c] = 1.0

    tensor[12, :, :] = float(board.turn)  # 1=white
    tensor[13, :, :] = float(board.has_kingside_castling_rights(chess.WHITE))
    tensor[14, :, :] = float(board.has_queenside_castling_rights(chess.WHITE))
    tensor[15, :, :] = float(board.has_kingside_castling_rights(chess.BLACK))
    tensor[16, :, :] = float(board.has_queenside_castling_rights(chess.BLACK))

    if board.ep_square is not None:
        r, c = divmod(board.ep_square, 8)
        tensor[17, r, c] = 1.0

    return tensor


def move_to_index(move):
    """Encode a move as an integer index (from_sq * 64 + to_sq)."""
    return move.from_square * 64 + move.to_square


def index_to_move(idx, board):
    """Decode an index back to a Move (if legal)."""
    from_sq = idx // 64
    to_sq = idx % 64
    move = chess.Move(from_sq, to_sq)
    # Handle promotion
    if move in board.legal_moves:
        return move
    # Try queen promotion
    move_promo = chess.Move(from_sq, to_sq, promotion=chess.QUEEN)
    if move_promo in board.legal_moves:
        return move_promo
    return None


# Test encoding
board = chess.Board()
t = board_to_tensor(board)
print(f"Board tensor shape: {t.shape}")
print(f"White pawns plane (channel 0):\n{t[0]}")

In [ ]:
class ResidualBlock(nn.Module):
    """A single residual block: Conv -> BN -> ReLU -> Conv -> BN + skip."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class ChessNet(nn.Module):
    """
    AlphaZero-style network:
      Input  : (batch, 18, 8, 8)
      Output : value scalar in (-1, 1), policy logits over 4096 actions
    """
    def __init__(self, num_res_blocks=10, channels=128):
        super().__init__()
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(18, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU()
        )
        # Residual tower
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(channels) for _ in range(num_res_blocks)]
        )
        # Value head
        self.value_head = nn.Sequential(
            nn.Conv2d(channels, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Tanh()  # Output in (-1, 1)
        )
        # Policy head
        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 2, 1, bias=False),
            nn.BatchNorm2d(2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128, NUM_ACTIONS)  # 4096 possible moves
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.res_blocks(x)
        value  = self.value_head(x)          # (batch, 1)
        policy = self.policy_head(x)         # (batch, 4096) — raw logits
        return value, policy


# Instantiate and test
model = ChessNet(num_res_blocks=5, channels=64).to(DEVICE)  # Small for demo
dummy = torch.zeros(1, 18, 8, 8).to(DEVICE)
value, policy = model(dummy)
print(f"Value output shape : {value.shape}  → {value.item():.4f}")
print(f"Policy output shape: {policy.shape}")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters   : {total_params:,}")

In [ ]:
def get_policy_probs(model, board, device=DEVICE):
    """
    Run the neural net on `board` and return a dict:
      {move: probability}  only for legal moves.
    """
    model.eval()
    with torch.no_grad():
        tensor = torch.tensor(board_to_tensor(board)).unsqueeze(0).to(device)
        value, policy_logits = model(tensor)

    policy_logits = policy_logits.squeeze(0).cpu().numpy()  # (4096,)

    # Mask illegal moves
    legal_moves = list(board.legal_moves)
    legal_indices = [move_to_index(m) for m in legal_moves]
    legal_logits  = policy_logits[legal_indices]

    # Softmax over legal moves only
    legal_logits = legal_logits - legal_logits.max()  # numerical stability
    probs = np.exp(legal_logits)
    probs = probs / probs.sum()

    return {m: p for m, p in zip(legal_moves, probs)}, value.item()


# Demo
board = chess.Board()
probs, val = get_policy_probs(model, board)
top5 = sorted(probs.items(), key=lambda x: -x[1])[:5]
print(f"Position value: {val:.4f}")
print("Top 5 moves (untrained network — random):")
for move, prob in top5:
    print(f"  {board.san(move):8s}  {prob:.4f}")

## Phase 4 — Monte Carlo Tree Search (MCTS)

MCTS builds a search tree by repeatedly:
1. **Select** — traverse tree using UCB score (exploitation + exploration)
2. **Expand** — add new node, evaluate with neural network
3. **Backup** — propagate value back up the tree

After many simulations, pick the move with the most visits.

In [ ]:
import math
from copy import deepcopy

C_PUCT = 1.5  # Exploration constant


class MCTSNode:
    def __init__(self, board, parent=None, prior=0.0):
        self.board    = board
        self.parent   = parent
        self.prior    = prior       # P(s, a) from policy network
        self.children = {}          # move -> MCTSNode
        self.N        = 0           # visit count
        self.W        = 0.0         # total value
        self.Q        = 0.0         # mean value W/N

    @property
    def is_leaf(self):
        return len(self.children) == 0

    def ucb_score(self, parent_N):
        """PUCT formula used in AlphaZero."""
        u = C_PUCT * self.prior * math.sqrt(parent_N) / (1 + self.N)
        return self.Q + u


class MCTS:
    def __init__(self, model, device=DEVICE):
        self.model  = model
        self.device = device

    def search(self, board, num_simulations=100):
        """Run MCTS and return move probabilities."""
        root = MCTSNode(deepcopy(board))

        for _ in range(num_simulations):
            node = root
            # ── 1. SELECT ──────────────────────────────────────
            while not node.is_leaf and not node.board.is_game_over():
                best_move, best_child = max(
                    node.children.items(),
                    key=lambda kv: kv[1].ucb_score(node.N)
                )
                node = best_child

            # ── 2. EXPAND ──────────────────────────────────────
            if not node.board.is_game_over():
                probs, value = get_policy_probs(self.model, node.board, self.device)
                for move, prob in probs.items():
                    child_board = deepcopy(node.board)
                    child_board.push(move)
                    node.children[move] = MCTSNode(child_board, parent=node, prior=prob)
            else:
                # Terminal node — get result
                result = node.board.result()
                value = 1.0 if result == '1-0' else (-1.0 if result == '0-1' else 0.0)
                if not node.board.turn:  # Flip for current player
                    value = -value

            # ── 3. BACKUP ──────────────────────────────────────
            while node is not None:
                node.N += 1
                node.W += value
                node.Q  = node.W / node.N
                value   = -value  # Flip for parent (opponent's perspective)
                node    = node.parent

        # Return visit-count based move probabilities
        total_visits = sum(c.N for c in root.children.values())
        if total_visits == 0:
            return {m: 1.0 / len(root.children) for m in root.children}
        return {m: c.N / total_visits for m, c in root.children.items()}

    def best_move(self, board, num_simulations=100):
        """Return the single best move."""
        probs = self.search(board, num_simulations)
        return max(probs, key=probs.get)


# Test MCTS
mcts = MCTS(model)
board = chess.Board()
print("Running 50 MCTS simulations...")
move_probs = mcts.search(board, num_simulations=50)
top5 = sorted(move_probs.items(), key=lambda x: -x[1])[:5]
print("Top 5 moves by MCTS visit count (untrained):")
for m, p in top5:
    print(f"  {board.san(m):8s}  {p:.4f}")

## Phase 5 — Self-Play Training Loop

This is the heart of AlphaZero:

1. **Generate data** — play games using MCTS, record `(state, policy, outcome)` triples
2. **Train** — update network to predict the recorded policies and outcomes
3. **Repeat** — the network keeps improving

### Loss Function
```
Loss = MSE(value_pred, game_outcome) + CrossEntropy(policy_pred, mcts_policy)
```

In [ ]:
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim


class ChessDataset(Dataset):
    """Dataset of (board_tensor, policy_target, value_target) triples."""
    def __init__(self, data):
        self.data = data  # list of (tensor, policy_vec, value)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tensor, policy, value = self.data[idx]
        return (
            torch.tensor(tensor, dtype=torch.float32),
            torch.tensor(policy, dtype=torch.float32),
            torch.tensor([value], dtype=torch.float32)
        )


def self_play_game(model, mcts, num_simulations=50, max_moves=200):
    """
    Play one game of self-play.
    Returns list of (board_tensor, policy_vector, placeholder_value).
    Values are filled in after the game ends.
    """
    board = chess.Board()
    game_data = []  # (tensor, policy_vec, turn)

    while not board.is_game_over() and board.fullmove_number <= max_moves:
        tensor = board_to_tensor(board)  # (18, 8, 8)

        # Get MCTS move probabilities
        move_probs = mcts.search(board, num_simulations)

        # Build policy vector (4096-dim)
        policy_vec = np.zeros(NUM_ACTIONS, dtype=np.float32)
        for move, prob in move_probs.items():
            policy_vec[move_to_index(move)] = prob

        # Store whose turn it is
        game_data.append((tensor, policy_vec, board.turn))

        # Sample move (add temperature for exploration in early game)
        moves  = list(move_probs.keys())
        probs  = np.array(list(move_probs.values()))
        chosen = np.random.choice(len(moves), p=probs)
        board.push(moves[chosen])

    # Assign outcomes
    result = board.result()
    if result == '1-0':
        outcome = 1.0
    elif result == '0-1':
        outcome = -1.0
    else:
        outcome = 0.0

    # Flip outcome relative to the player who moved
    final_data = []
    for tensor, policy_vec, turn in game_data:
        value = outcome if turn == chess.WHITE else -outcome
        final_data.append((tensor, policy_vec, value))

    return final_data


def train_epoch(model, optimizer, dataloader, device=DEVICE):
    """One training epoch."""
    model.train()
    total_loss = 0.0

    for batch_tensors, batch_policies, batch_values in dataloader:
        batch_tensors  = batch_tensors.to(device)
        batch_policies = batch_policies.to(device)
        batch_values   = batch_values.to(device)

        optimizer.zero_grad()
        pred_values, pred_policy_logits = model(batch_tensors)

        # Value loss: MSE
        value_loss = F.mse_loss(pred_values, batch_values)

        # Policy loss: Cross-entropy (target = MCTS probs)
        log_probs    = F.log_softmax(pred_policy_logits, dim=1)
        policy_loss  = -(batch_policies * log_probs).sum(dim=1).mean()

        loss = value_loss + policy_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

print("Functions defined. Ready to train!")

In [ ]:
import matplotlib.pyplot as plt

# ─── Hyperparameters ───────────────────────────────────────────────────────
NUM_SELF_PLAY_GAMES  = 5       # Increase to 100+ for real training
MCTS_SIMULATIONS     = 20      # Increase to 400+ for stronger play
BATCH_SIZE           = 32
LEARNING_RATE        = 1e-3
NUM_EPOCHS           = 3
# ───────────────────────────────────────────────────────────────────────────

model     = ChessNet(num_res_blocks=5, channels=64).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
mcts      = MCTS(model)

all_losses = []

print("=" * 60)
print("Starting self-play training loop (mini demo)")
print("=" * 60)

for iteration in range(1, 4):  # 3 iterations for demo
    print(f"\n🔄 Iteration {iteration}")

    # ── Step 1: Self-play data generation ──
    print(f"  Generating {NUM_SELF_PLAY_GAMES} self-play games...")
    replay_buffer = []
    for g in tqdm(range(NUM_SELF_PLAY_GAMES), desc="  Self-play"):
        game_data = self_play_game(model, mcts, num_simulations=MCTS_SIMULATIONS)
        replay_buffer.extend(game_data)
    print(f"  Collected {len(replay_buffer)} training positions")

    # ── Step 2: Train ──
    dataset    = ChessDataset(replay_buffer)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"  Training for {NUM_EPOCHS} epochs...")
    for epoch in range(1, NUM_EPOCHS + 1):
        loss = train_epoch(model, optimizer, dataloader)
        all_losses.append(loss)
        print(f"    Epoch {epoch}: loss = {loss:.4f}")

print("\n✅ Training complete!")

# Plot losses
plt.figure(figsize=(8, 4))
plt.plot(all_losses, 'b-o', markersize=4)
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Self-Play Training Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

## 🤖 Play Against Your AI!

In [ ]:
def play_vs_ai(model, mcts_sims=100, you_play='white'):
    """
    Interactive game vs the AI.
    Enter moves in UCI format (e.g. 'e2e4') or SAN (e.g. 'e4').
    Type 'quit' to stop.
    """
    board = chess.Board()
    human_color = chess.WHITE if you_play == 'white' else chess.BLACK
    mcts = MCTS(model)

    while not board.is_game_over():
        display(SVG(chess.svg.board(board, size=350)))
        print(f"\nTurn: {'White' if board.turn else 'Black'}")

        if board.turn == human_color:
            # Human move
            while True:
                user_input = input("Your move (UCI or SAN): ").strip()
                if user_input.lower() == 'quit':
                    print("Game aborted.")
                    return
                try:
                    # Try UCI first, then SAN
                    try:
                        move = chess.Move.from_uci(user_input)
                    except:
                        move = board.parse_san(user_input)
                    if move in board.legal_moves:
                        board.push(move)
                        break
                    else:
                        print("Illegal move. Try again.")
                except Exception as e:
                    print(f"Invalid input: {e}. Try again.")
        else:
            # AI move
            print("AI is thinking...")
            ai_move = mcts.best_move(board, num_simulations=mcts_sims)
            print(f"AI plays: {board.san(ai_move)}")
            board.push(ai_move)

    display(SVG(chess.svg.board(board, size=350)))
    print(f"\n🏁 Game over! Result: {board.result()}")
    print(board.result())

# Uncomment to play!
# play_vs_ai(model, mcts_sims=50, you_play='white')

## 💾 Save & Load Model

In [ ]:
# Save
torch.save({
    'model_state_dict':     model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, 'chess_ai_checkpoint.pt')
print("Model saved to chess_ai_checkpoint.pt")

# Load
checkpoint = torch.load('chess_ai_checkpoint.pt', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
print("Model loaded successfully!")

## 🚀 Next Steps to Strengthen Your Engine

| Improvement | Impact | Effort |
|---|---|---|
| More residual blocks (10–20) | 🔥 High | Low |
| More MCTS simulations (400+) | 🔥 High | Low |
| More self-play games (1000+) | 🔥 High | Medium |
| Dirichlet noise in root MCTS | Medium | Low |
| Temperature annealing | Medium | Low |
| Larger replay buffer | Medium | Medium |
| Supervised pretraining on human games | 🔥 High | Medium |
| Distributed self-play (multi-GPU) | Very High | High |

### 📚 Resources
- [Leela Chess Zero](https://lczero.org/) — open-source AlphaZero clone
- [AlphaZero paper](https://arxiv.org/abs/1712.01815) — original DeepMind paper
- [python-chess docs](https://python-chess.readthedocs.io/) — chess library reference
- [KataGo](https://github.com/lightvector/KataGo) — many AlphaZero improvements explained